In [12]:
!pip install open-clip-torch

In [13]:
import concurrent.futures
import csv
import glob
import os
from typing import List

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
import open_clip

class ImageEncoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, images):
        return self.model(images)


def preprocess_frame(frame, transform):
    pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    return transform(pil_image).unsqueeze(0)  # (1, C, H, W)


@torch.no_grad()
def encode_batch(frames, model, transform, device):
    with concurrent.futures.ThreadPoolExecutor() as ex:
        processed = list(ex.map(lambda fr: preprocess_frame(fr, transform), frames))

    images = torch.cat(processed, dim=0).to(device, non_blocking=True)

    with torch.amp.autocast(device_type="cuda", enabled=device.type == "cuda"):
        feats = model(images)

    feats = F.normalize(feats.float(), dim=-1)
    return feats


def save_image_webp(img_bgr, path: str, quality: int = 80, resize_factor: float = 0.5):
    if resize_factor != 1.0:
        img_bgr = cv2.resize(img_bgr, (0, 0), fx=resize_factor, fy=resize_factor)
    img_pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    img_pil.save(path, format="WEBP", quality=quality)
    
def save_feature_tensor(tensor: torch.Tensor, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(tensor.cpu(), path)


def process_video(
    video_path: str,
    kf_dir: str,
    maps_dir: str,
    model,
    transform,
    feature_dir: str,
    device: torch.device,
    sim_threshold: float,
    skip_frames: int,
    batch_size: int = 64,
    resize_factor: float = 1,
    webp_quality: int = 80,
) -> int:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    video_name = os.path.splitext(os.path.basename(video_path))[0]
    map_csv = os.path.join(maps_dir, f"{video_name}_map.csv")

    frames: List = []
    indices: List[int] = []
    keyframe_count = 0
    prev_feat = None

    with open(map_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["FrameID", "Seconds"])

        pbar = tqdm(total=total_frames if total_frames > 0 else None, desc=f"Processing {video_name}")
        frame_id = 0

        while True:
            ok, frame = cap.read()
            if not ok:
                break

            # process every (skip_frames + 1)-th frame
            if (frame_id % (skip_frames + 1) == 0):
                frames.append(frame)
                indices.append(frame_id)

                if len(frames) == batch_size:
                    feats = encode_batch(frames, model, transform, device)
                    for img, fid, feat in zip(frames, indices, feats):
                        if prev_feat is None:
                            # always take the very first considered frame
                            kf_path = os.path.join(kf_dir, f"keyframe_{fid}.webp")
                            save_image_webp(img, kf_path, quality=webp_quality, resize_factor=resize_factor)
                            feature_path = os.path.join(feature_dir, f"keyframe_{fid}.pt")
                            save_feature_tensor(feat, feature_path)
                            writer.writerow([fid, f"{fid / fps:.2f}"])
                            prev_feat = feat
                            keyframe_count += 1
                        else:
                            sim = torch.dot(prev_feat, feat).item()  # cosine since normalized
                            if sim < sim_threshold:
                                # only save keyframes have low similarity score with their previous frame
                                kf_path = os.path.join(kf_dir, f"keyframe_{fid}.webp")
                                save_image_webp(img, kf_path, quality=webp_quality, resize_factor=resize_factor)
                                feature_path = os.path.join(feature_dir, f"keyframe_{fid}.pt")
                                save_feature_tensor(feat, feature_path)
                                writer.writerow([fid, f"{fid / fps:.2f}"])
                                prev_feat = feat
                                keyframe_count += 1

                    frames.clear()
                    indices.clear()

            frame_id += 1
            pbar.update(1)

        # flush leftover
        if frames:
            feats = encode_batch(frames, model, transform, device)
            for img, fid, feat in zip(frames, indices, feats):
                if prev_feat is None:
                    kf_path = os.path.join(kf_dir, f"keyframe_{fid}.webp")
                    save_image_webp(img, kf_path, quality=webp_quality, resize_factor=resize_factor)
                    feature_path = os.path.join(feature_dir, f"keyframe_{fid}.pt")
                    save_feature_tensor(feat, feature_path)
                    writer.writerow([fid, f"{fid / fps:.2f}"])
                    prev_feat = feat
                    keyframe_count += 1
                else:
                    sim = torch.dot(prev_feat, feat).item()
                    if sim < sim_threshold:
                        kf_path = os.path.join(kf_dir, f"keyframe_{fid}.webp")
                        save_image_webp(img, kf_path, quality=webp_quality, resize_factor=resize_factor)
                        feature_path = os.path.join(feature_dir, f"keyframe_{fid}.pt")
                        save_feature_tensor(feat, feature_path)
                        writer.writerow([fid, f"{fid / fps:.2f}"])
                        prev_feat = feat
                        keyframe_count += 1

        pbar.close()
    cap.release()
    return keyframe_count

def process_all_videos(
    input_folder: str,
    output_base: str,
    sim_threshold: float,
    skip_frames: int,
    batch_size: int,
    resize_factor: float,
    webp_quality: int,
    start_index: int,
    pattern: str,
    device: torch.device,
):
    # Load OpenCLIP once
    model, _, transform = open_clip.create_model_and_transforms(
        "ViT-H-14-378-quickgelu", pretrained="dfn5b"
    )

    model = ImageEncoder(model).to(device)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    
    model.eval()

    # discover videos
    video_files = sorted(glob.glob(os.path.join(input_folder, pattern)))
    
    if not video_files:
        print(f"No videos matched: {os.path.join(input_folder, pattern)}")
    
    maps_dir = os.path.join(output_base, "maps")
    os.makedirs(maps_dir, exist_ok=True)

    kf_folder = os.path.join(output_base, "keyframe")
    os.makedirs(kf_folder, exist_ok=True)
    
    embedding_folder = os.path.join(output_base, "embedding")
    os.makedirs(embedding_folder, exist_ok=True)
    
    for video_path in video_files[start_index:]:
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        kf_dir = os.path.join(kf_folder, video_name)
        os.makedirs(kf_dir, exist_ok=True)
        feature_dir = os.path.join(embedding_folder, video_name)
        os.makedirs(feature_dir, exist_ok=True)
        
        print(f"Processing video: {video_name}")
        
        try:
            kf_count = process_video(
                video_path=video_path,
                kf_dir=kf_dir,
                feature_dir=feature_dir,
                maps_dir=maps_dir,
                model=model,
                transform=transform,
                device=device,
                sim_threshold=sim_threshold,
                skip_frames=skip_frames,
                batch_size=batch_size,
                resize_factor=resize_factor,
                webp_quality=webp_quality,
            )
            print(f"Total keyframes: {kf_count}")
        except Exception as e:
            print(f"Error processing {video_name}: {e}")
        print("-" * 20)

In [14]:
device = torch.device("cpu" if not torch.cuda.is_available() else "cuda")
process_all_videos(
    input_folder="/kaggle/input/datasets/dtdat1725/aic-hcmc-2025-batch-1/video/L21_a",
    output_base="data",
    sim_threshold=0.95,
    skip_frames=5,
    batch_size=128,
    resize_factor=0.5,
    webp_quality=80,
    start_index=-1,
    pattern="*.mp4",
    device=device,
)

Using 2 GPUs
Processing video: L21_V031



Processing L21_V031: 100%|██████████| 27528/27528 [05:01<00:00, 91.28it/s] 

Total keyframes: 593
--------------------


In [15]:
!zip -r results.zip /kaggle/working
from IPython.display import FileLink
FileLink(r'results.zip')

updating: kaggle/working/ (stored 0%)
updating: kaggle/working/.virtual_documents/ (stored 0%)
updating: kaggle/working/.virtual_documents/__notebook_source__.ipynb (deflated 72%)
updating: kaggle/working/data/ (stored 0%)
updating: kaggle/working/data/embedding/ (stored 0%)
updating: kaggle/working/data/embedding/L21_V031/ (stored 0%)
updating: kaggle/working/data/embedding/L21_V031/keyframe_1500.pt (deflated 26%)
updating: kaggle/working/data/embedding/L21_V031/keyframe_1302.pt (deflated 26%)
updating: kaggle/working/data/embedding/L21_V031/keyframe_1248.pt (deflated 26%)
updating: kaggle/working/data/embedding/L21_V031/keyframe_366.pt (deflated 26%)
updating: kaggle/working/data/embedding/L21_V031/keyframe_18.pt (deflated 26%)
updating: kaggle/working/data/embedding/L21_V031/keyframe_294.pt (deflated 26%)
updating: kaggle/working/data/embedding/L21_V031/keyframe_480.pt (deflated 26%)
updating: kaggle/working/data/embedding/L21_V031/keyframe_1434.pt (deflated 26%)
updating: kaggle/wo

/kaggle/working/results.zip